In [ ]:
!sudo apt update
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
#Check this site for the latest download link https://www.apache.org/dyn/closer.lua/spark/spark-3.2.1/spark-3.2.1-bin-hadoop3.2.tgz
!wget -q https://dlcdn.apache.org/spark/spark-3.2.1/spark-3.2.1-bin-hadoop3.2.tgz
!tar xf spark-3.2.1-bin-hadoop3.2.tgz
!pip install -q findspark
!pip install pyspark
!pip install py4j

import os
import sys
# os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
# os.environ["SPARK_HOME"] = "/content/spark-3.2.1-bin-hadoop3.2"


import findspark
findspark.init()
findspark.find()

import pyspark

from pyspark.sql import DataFrame, SparkSession
from typing import List
import pyspark.sql.types as T
import pyspark.sql.functions as F

spark= SparkSession \
       .builder \
       .appName("Primer Examen") \
       .getOrCreate()

spark

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [4,685 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:4 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,578 B]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:7 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:8 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [113 kB]
Get:9 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,920 kB]
Get:10 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.8 MB]
Get:12 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-securi

In [ ]:

import requests
import pandas as pd
from google.colab import data_table  # Para usar DataTable en Colab
import matplotlib.pyplot as plt


# Descargar el archivo CSV desde la URL -> ETL
path = "https://github.com/reisanar/datasets/raw/master/HollywoodMovies.csv"
req = requests.get(path) # Extrae
url_content = req.content # CSV Transformación
name_file = "HollywoodMovies.csv" # Gobierno
csv_file = open(name_file, "wb")
csv_file.write(url_content) # Similar a la transmisión

#Mover archivo
import os
import shutil

os.makedirs('/content/staging/', exist_ok=True)
shutil.move('/content/HollywoodMovies.csv','/content/staging/HollywoodMovies.csv')


'/content/staging/HollywoodMovies.csv'

In [ ]:
#Ingesta a Master
spark.read.csv('/content/staging/HollywoodMovies.csv', header=True, inferSchema=True)\
  .write.mode("overwrite").parquet("/content/master/HollywoodMovies")

In [ ]:
#Analisis exploratorio
df_peliculas = spark.read.parquet("/content/master/HollywoodMovies")


df_peliculas.printSchema()

root
 |-- Movie: string (nullable = true)
 |-- LeadStudio: string (nullable = true)
 |-- RottenTomatoes: integer (nullable = true)
 |-- AudienceScore: integer (nullable = true)
 |-- Story: string (nullable = true)
 |-- Genre: string (nullable = true)
 |-- TheatersOpenWeek: integer (nullable = true)
 |-- OpeningWeekend: double (nullable = true)
 |-- BOAvgOpenWeekend: integer (nullable = true)
 |-- DomesticGross: double (nullable = true)
 |-- ForeignGross: double (nullable = true)
 |-- WorldGross: double (nullable = true)
 |-- Budget: double (nullable = true)
 |-- Profitability: double (nullable = true)
 |-- OpenProfit: double (nullable = true)
 |-- Year: integer (nullable = true)



In [ ]:
df_peliculas.count()

970

In [ ]:
from google.colab import data_table
import pandas as pd
df_pandas = df_peliculas.toPandas()
data_table.DataTable(df_pandas, include_index=False, num_rows_per_page=10)

,Movie,LeadStudio,RottenTomatoes,AudienceScore,Story,Genre,TheatersOpenWeek,OpeningWeekend,BOAvgOpenWeekend,DomesticGross,ForeignGross,WorldGross,Budget,Profitability,OpenProfit,Year
0,Spider-Man 3,Sony,61.0,54.0,Metamorphosis,Action,4252.0,151.10,35540.0,336.53,554.34,890.87,258.0,345.30,58.57,2007
1,Shrek the Third,Paramount,42.0,57.0,Quest,Animation,4122.0,121.60,29507.0,322.72,476.24,798.96,160.0,499.35,76.00,2007
2,Transformers,Paramount,57.0,89.0,Monster Force,Action,4011.0,70.50,17577.0,319.25,390.46,709.71,150.0,473.14,47.00,2007
3,Pirates of the Caribbean: At World's End,Disney,45.0,74.0,Rescue,Action,4362.0,114.70,26302.0,309.42,654.00,963.42,300.0,321.14,38.23,2007
4,Harry Potter and the Order of the Phoenix,Warner Bros,78.0,82.0,Quest,Adventure,4285.0,77.10,17998.0,292.00,647.88,939.89,150.0,626.59,51.40,2007
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
965,The Canyons,IFC,22.0,NaN,None,None,1.0,0.01,13351.0,0.06,0.14,0.19,NaN,77.21,NaN,2013
966,The Call,TriStar,43.0,66.0,None,None,2507.0,17.12,6828.0,51.87,16.70,68.57,13.0,527.48,131.69,2013
967,The English Teacher,Cinedigm Entertainment,42.0,NaN,None,None,2.0,0.01,3001.0,0.10,0.06,0.10,NaN,NaN,NaN,2013
968,John Dies at the End,Magnolia,61.0,53.0,None,None,1.0,0.01,12467.0,0.14,NaN,0.14,1.0,14.20,1.00,2013


In [ ]:
df_peliculas.select("LeadStudio").distinct().show(100,truncate=False)

+------------------------+
|LeadStudio              |
+------------------------+
|Relativity Media        |
|Entertainment One       |
|TriStar                 |
|Pixar                   |
|Focus                   |
|Crest                   |
|Village Roadshow        |
|LD Entertainment        |
|Sony                    |
|Yash Raj                |
|Oscillloscope           |
|Weinstein               |
|Liberty Starz           |
|Aardman Animations      |
|Virgin                  |
|Roadside Attractions    |
|Warner Bros             |
|DreamWorks              |
|Vertigo                 |
|Paramount               |
|ARC Entertainment       |
|Lionsgate               |
|Cohen Media             |
|Overture                |
|Spyglass Entertainment  |
|Millenium Entertainment |
|Rocky Mountain          |
|Happy Madison           |
|Open Road               |
|Music Box Films         |
|Summit                  |
|Regency Enterprises     |
|Samuel Goldwyn          |
|Mediaplex               |
|

#Ejercicio 1
##Encontrar las películas más rentables por estudio


In [ ]:
import pyspark.sql.functions as F
def obtieneEstudio(estudio):
  return spark.read.option("basePath","/content/master/PeliculasPorEstudio/")\
    .parquet("/content/master/PeliculasPorEstudio/LeadStudio="+estudio)\
    .select("LeadStudio","Profitability")\
    .orderBy(F.col("Profitability").desc())
    #Regresa un DF, este DF esta ordenado por rentabilidad


In [ ]:
# Encontrar las películas más rentables por cada estudio
df_peliculas.write.mode("overwrite").partitionBy("LeadStudio").parquet("/content/master/PeliculasPorEstudio")
estudios = [
    "Relativity Media",
    "Entertainment One",
    "TriStar",
    "Pixar"
]


# Mostrar los resultados
for estudio in estudios:
  df_estudio = obtieneEstudio(estudio)
  df_estudio.show(3)
  #obtieneEstudio(estudio).show(3)

+----------------+-------------+
|      LeadStudio|Profitability|
+----------------+-------------+
|Relativity Media|       915.44|
|Relativity Media|       887.33|
|Relativity Media|       507.51|
+----------------+-------------+
only showing top 3 rows
+-----------------+-------------+
|       LeadStudio|Profitability|
+-----------------+-------------+
|Entertainment One|         NULL|
+-----------------+-------------+

+----------+-------------+
|LeadStudio|Profitability|
+----------+-------------+
|   TriStar|       588.36|
|   TriStar|       573.78|
|   TriStar|       527.48|
+----------+-------------+
only showing top 3 rows
+----------+-------------+
|LeadStudio|Profitability|
+----------+-------------+
|     Pixar|       279.93|
+----------+-------------+



In [ ]:
# Otra forma de resolver
estudios = [
    "Relativity Media",
    "Entertainment One",
    "TriStar",
    "Pixar"
]


# Mostrar los resultados
for estudio in estudios:
  df_peliculas.where(F.col("LeadStudio")==estudio).select("LeadStudio","Profitability")\
    .orderBy(F.col("Profitability").desc()).show(3)

+----------------+-------------+
|      LeadStudio|Profitability|
+----------------+-------------+
|Relativity Media|       915.44|
|Relativity Media|       887.33|
|Relativity Media|       507.51|
+----------------+-------------+
only showing top 3 rows
+-----------------+-------------+
|       LeadStudio|Profitability|
+-----------------+-------------+
|Entertainment One|         NULL|
+-----------------+-------------+

+----------+-------------+
|LeadStudio|Profitability|
+----------+-------------+
|   TriStar|       588.36|
|   TriStar|       573.78|
|   TriStar|       527.48|
+----------+-------------+
only showing top 3 rows
+----------+-------------+
|LeadStudio|Profitability|
+----------+-------------+
|     Pixar|       279.93|
+----------+-------------+



In [ ]:
#Otra forma de resolverlo
from pyspark.sql.window import Window

estudios = [
    "Relativity Media",
    "Entertainment One",
    "TriStar",
    "Pixar"
]

win = Window.orderBy(F.col("Profitability").desc())
# Mostrar los resultados
for estudio in estudios:
  df_con_numero = df_peliculas.where(F.col("LeadStudio")==estudio).select("LeadStudio","Profitability")\
    .orderBy(F.col("Profitability").desc()).withColumn("Top3",F.row_number().over(win))
  df_con_numero.where(F.col("Top3").isin("1","2","3")).show()

+----------------+-------------+----+
|      LeadStudio|Profitability|Top3|
+----------------+-------------+----+
|Relativity Media|       915.44|   1|
|Relativity Media|       887.33|   2|
|Relativity Media|       507.51|   3|
+----------------+-------------+----+

+-----------------+-------------+----+
|       LeadStudio|Profitability|Top3|
+-----------------+-------------+----+
|Entertainment One|         NULL|   1|
+-----------------+-------------+----+

+----------+-------------+----+
|LeadStudio|Profitability|Top3|
+----------+-------------+----+
|   TriStar|       588.36|   1|
|   TriStar|       573.78|   2|
|   TriStar|       527.48|   3|
+----------+-------------+----+

+----------+-------------+----+
|LeadStudio|Profitability|Top3|
+----------+-------------+----+
|     Pixar|       279.93|   1|
+----------+-------------+----+



In [ ]:
# Encontrar las películas más rentables por cada estudio
df_peliculas.write.mode("overwrite").partitionBy("LeadStudio").parquet("/content/master/PeliculasPorEstudio")

# Extraer automáticamente la lista de estudios únicos (filtrando nulos si los hay)
estudios = [
    row.LeadStudio
    for row in df_peliculas.select("LeadStudio").dropna().distinct().collect()
]

# Mostrar los resultados
for estudio in estudios:
    df_estudio = obtieneEstudio(estudio)
    df_estudio.show(3)

+----------------+-------------+
|      LeadStudio|Profitability|
+----------------+-------------+
|Relativity Media|       915.44|
|Relativity Media|       887.33|
|Relativity Media|       507.51|
+----------------+-------------+
only showing top 3 rows
+-----------------+-------------+
|       LeadStudio|Profitability|
+-----------------+-------------+
|Entertainment One|         NULL|
+-----------------+-------------+

+----------+-------------+
|LeadStudio|Profitability|
+----------+-------------+
|   TriStar|       588.36|
|   TriStar|       573.78|
|   TriStar|       527.48|
+----------+-------------+
only showing top 3 rows
+----------+-------------+
|LeadStudio|Profitability|
+----------+-------------+
|     Pixar|       279.93|
+----------+-------------+

+----------+-------------+
|LeadStudio|Profitability|
+----------+-------------+
|     Focus|        616.9|
|     Focus|       383.95|
|     Focus|       236.57|
+----------+-------------+
only showing top 3 rows
+----------+

# Ejercicio 2
Calcular la media de puntuación de audiencia y crítica por género


In [ ]:
from pyspark.sql import functions as F

# Cálculo de promedios por género (ajusta los nombres de las columnas de puntuación según tu schema)
df_medias = df_peliculas.groupBy("Genre").agg(
    F.round(F.avg("AudienceScore"), 2).alias("Media_Audiencia"),
    F.round(F.avg("RottenTomatoes"), 2).alias("Media_Critica")  # O "CriticScore" / "Rotten Tomatoes %"
)

df_medias.show()

+-----------+---------------+-------------+
|      Genre|Media_Audiencia|Media_Critica|
+-----------+---------------+-------------+
|      Crime|          62.53|        50.93|
|    Romance|          65.68|        50.68|
|   Thriller|          66.54|        60.91|
|  Adventure|          63.47|        54.27|
|       NULL|          63.24|        56.85|
|      Drama|          65.41|        57.16|
|Documentary|           72.2|         63.4|
|    Fantasy|           72.0|        62.33|
|    Mystery|           55.0|         42.0|
|    Musical|          75.75|         61.5|
|  Animation|          66.29|        62.14|
|     Horror|          49.27|        37.39|
|  Biography|           72.0|        72.71|
|     Comedy|          56.39|        44.65|
|     Action|           59.3|        44.75|
+-----------+---------------+-------------+



# Ejercicio 3
Identificar películas con gran diferencia entre crítica y audiencia


In [ ]:
from pyspark.sql import functions as F

# 1. Calcular la diferencia absoluta entre Audiencia y Crítica
df_diferencia = df_peliculas.withColumn(
    "Diferencia",
    F.abs(F.col("AudienceScore") - F.col("RottenTomatoes"))
)

# 2. Seleccionar columnas corrigiendo 'Film' por 'Movie'
df_discrepancia = df_diferencia.select(
    "Movie", "Genre", "AudienceScore", "RottenTomatoes", "Diferencia"
).orderBy(F.col("Diferencia").desc())

# 3. Mostrar el Top 10 de películas con mayor brecha
df_discrepancia.show(10, truncate=False)

+-----------------------------------+-------+-------------+--------------+----------+
|Movie                              |Genre  |AudienceScore|RottenTomatoes|Diferencia|
+-----------------------------------+-------+-------------+--------------+----------+
|Daddy Day Camp                     |Comedy |63           |1             |62        |
|P.S. I Love You                    |Romance|82           |21            |61        |
|I Now Pronounce You Chuck and Larry|Comedy |73           |13            |60        |
|Wild Hogs                          |Comedy |72           |14            |58        |
|Good Luck Chuck                    |Comedy |61           |3             |58        |
|Jobs                               |NULL   |85           |27            |58        |
|Stomp the Yard                     |Musical|83           |27            |56        |
|Transformers: Revenge of the Fallen|Action |76           |20            |56        |
|Safe Haven                         |NULL   |68       

# Ejercicio 4
Listar las películas con presupuestos superiores a $150 millones

In [ ]:
from pyspark.sql import functions as F

# Filtrar películas con presupuesto mayor a 150 millones y ordenar de mayor a menor
df_presupuesto_alto = df_peliculas.filter(F.col("Budget") > 150) \
                                  .select("Movie", "Genre", "LeadStudio", "Budget", "Year") \
                                  .orderBy(F.col("Budget").desc())

df_presupuesto_alto.show(truncate=False)

+------------------------------------------+---------+-----------+------+----+
|Movie                                     |Genre    |LeadStudio |Budget|Year|
+------------------------------------------+---------+-----------+------+----+
|Pirates of the Caribbean: At World's End  |Action   |Disney     |300.0 |2007|
|Tangled                                   |Animation|Disney     |260.0 |2010|
|Spider-Man 3                              |Action   |Sony       |258.0 |2007|
|Harry Potter and the Half-Blood Prince    |Adventure|Warner Bros|250.0 |2009|
|Pirates of the Caribbean:On Stranger Tides|Action   |Disney     |250.0 |2011|
|John Carter                               |NULL     |Buena Vista|250.0 |2012|
|The Dark Knight Rises                     |NULL     |Warner Bros|250.0 |2012|
|Avatar                                    |Action   |Fox        |237.0 |2009|
|Quantum of Solace                         |Action   |MGM        |230.0 |2008|
|The Amazing Spider-Man                    |NULL    

# Ejercicio 5
Análisis de la rentabilidad de películas por año

In [ ]:
df_rentabilidad_anual = df_peliculas.groupBy("Year").agg(
    F.count("Movie").alias("Total_Peliculas"),
    F.round(F.avg("Profitability"), 2).alias("Rentabilidad_Promedio"),
    F.round(F.max("Profitability"), 2).alias("Max_Rentabilidad"),
    F.round(F.avg("WorldGross"), 2).alias("Recaudacion_Mundial_Promedio"),
    F.round(F.avg("Budget"), 2).alias("Presupuesto_Promedio")
).orderBy(F.col("Year").asc())

df_rentabilidad_anual.show(truncate=False)

+----+---------------+---------------------+----------------+----------------------------+--------------------+
|Year|Total_Peliculas|Rentabilidad_Promedio|Max_Rentabilidad|Recaudacion_Mundial_Promedio|Presupuesto_Promedio|
+----+---------------+---------------------+----------------+----------------------------+--------------------+
|2007|91             |355.15               |3085.48         |188.35                      |62.98               |
|2008|148            |379.68               |6694.4          |136.34                      |48.49               |
|2009|137            |298.76               |1419.64         |166.71                      |54.49               |
|2010|136            |394.55               |5917.03         |166.5                       |56.73               |
|2011|141            |373.51               |6467.27         |166.85                      |55.4                |
|2012|157            |567.02               |10175.85        |235.52                      |67.04         